# 04 - Feature Engineering en Spark (PySpark)

Pipeline de construcción de features sobre el dataset OULAD ejecutado en Databricks Community Edition. Se construye replicando la lógica de '02_features.ipynb' en PySpark lo que permite escalar el proceso a volúmenes mayores de datos sin cambiar la lógica de negocio.

- **Ruta de datos:** /Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/
- **Output:** df_modelo_spark.csv (27.553 filas × 25 columnas)
- **Justificación Spark:** arquitectura preparada para producción, no por volumen del CSV.

## 0. Setup y carga de tablas OULAD

Se cargan las 7 tablas del original desde el Workspace de Databricks. Se verifica, además, el número de filas contra los valores conocidos del dataset original para detectar posibles errores de lectura antes de procesar.

In [0]:
from pyspark.sql import functions as F

PATH = "/Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/"

courses         = spark.read.csv(PATH + "courses.csv",             header=True, inferSchema=True)
assessments     = spark.read.csv(PATH + "assessments.csv",         header=True, inferSchema=True)
vle             = spark.read.csv(PATH + "vle.csv",                 header=True, inferSchema=True)
student_info    = spark.read.csv(PATH + "studentInfo.csv",         header=True, inferSchema=True)
student_reg     = spark.read.csv(PATH + "studentRegistration.csv", header=True, inferSchema=True)
student_assess  = spark.read.csv(PATH + "studentAssessment.csv",   header=True, inferSchema=True)
student_vle     = spark.read.csv(PATH + "studentVle.csv",          header=True, inferSchema=True)

# Verificación
tablas = {
    "courses": (courses, 22),
    "assessments": (assessments, 206),
    "vle": (vle, 6364),
    "student_info": (student_info, 32593),
    "student_reg": (student_reg, 32593),
    "student_assess": (student_assess, 173912),
    "student_vle": (student_vle, 10655280),
}

for nombre, (df, esperado) in tablas.items():
    n = df.count()
    estado = "✅" if n == esperado else f"⚠️  esperado {esperado}"
    print(f"{estado}  {nombre}: {n:,} filas | {len(df.columns)} cols")

✅  courses: 22 filas | 3 cols
✅  assessments: 206 filas | 6 cols
✅  vle: 6,364 filas | 6 cols
✅  student_info: 32,593 filas | 12 cols
✅  student_reg: 32,593 filas | 5 cols
✅  student_assess: 173,912 filas | 5 cols
✅  student_vle: 10,655,280 filas | 6 cols


**Comentarios:**

El resultado de la primera carga es correcto, vemos que el resumen de datos coincide con lo esperado y visto ya en el dataset del EDA. La tabla con más registros es student_vle con 10.655.280 filas que tras la eliminación de duplicados exactos se quedará aproximadamente en 9.87 millones.

## 1. Construir df_activos

Comenzamos haciendo el join de student_info con student_registration para añadir date_unregistration y después incluimos el filtro para excluir las matrículas dadas de baja dentro de la ventana de predicción, añadimos la variable 'riesgo' al df para identificar de forma binaria riesgo = 0 y riesgo = 1, finalmente, corregimos el % que faltaba en el rango 10-20 de imd_band y finalmente, incorporamos una verificación para comprobar si el conteo de totales y el split de riesgo es el esperado.

In [0]:
# Join student_info + student_reg
df = student_info.join(
    student_reg.select("code_module", "code_presentation", "id_student", "date_unregistration"),
    on=["code_module", "code_presentation", "id_student"],
    how="left"
)

# Filtro de activos: excluir date_unregistration <= 27
df_activos = df.filter(
    F.col("date_unregistration").isNull() | (F.col("date_unregistration") > 27)
)

# Variable riesgo
df_activos = df_activos.withColumn(
    "riesgo",
    F.when(F.col("final_result").isin("Fail", "Withdrawn"), 1).otherwise(0)
)

# Corrección imd_band (misma que en pandas)
df_activos = df_activos.withColumn(
    "imd_band",
    F.when(F.col("imd_band") == "10-20", "10-20%")
     .when(F.col("imd_band").isNull(), "Missing")
     .otherwise(F.col("imd_band"))
)

# Verificación
n = df_activos.count()
n_riesgo = df_activos.filter(F.col("riesgo") == 1).count()
print(f"df_activos: {n:,} estudiantes")
print(f"Riesgo=1:   {n_riesgo:,} ({n_riesgo/n*100:.1f}%)")
print(f"Riesgo=0:   {n - n_riesgo:,} ({(n - n_riesgo)/n*100:.1f}%)")

df_activos: 27,553 estudiantes
Riesgo=1:   12,168 (44.2%)
Riesgo=0:   15,385 (55.8%)


**Comentarios:**

Los datos son exactamente los mismos que en el df_modelo.csv: 27.553 matrículas activas en el día 27, con un 44,2% de riesgo = 1, por lo que confirmamos que el filtrado coincide entre las dos implementaciones.

## 2. Dedup studentVle + features VLE

Procedemos a la limpieza de la tabla student_vle, eliminando los 787.170 registros exactos (mismo alumno, módulo, semestre, recurso, día y número de clics). Igual que en la primera ocasión se eliminan antes de filtrar para evitar inflar los datos de conteo de la actividad.

Marcamos el rango de la ventana de predicción en 0 a 27 días, agregando los días anteriores a 0 de manera separada como clics_precurso para conservar esa señal sin influir en la ventana principal.

In [0]:
import pandas as pd

# 1. Eliminar duplicados exactos (las 6 columnas idénticas)
student_vle_clean = student_vle.dropDuplicates()

# Verificación intermedia — debe dar ~9.868.110 (10.655.280 - 787.170)
print(f"Tras dropDuplicates: {student_vle_clean.count():,} filas")

# 2. Filtrar ventana 0-27 y pre-curso por separado
vle_ventana  = student_vle_clean.filter((F.col("date") >= 0) & (F.col("date") <= 27))
vle_precurso = student_vle_clean.filter(F.col("date") < 0)

# 3. Agregar clics por semana en ventana (groupBy suma los sum_click legítimos)
vle_agg = vle_ventana.groupBy("id_student", "code_module", "code_presentation").agg(
    F.sum("sum_click").alias("total_clics"),
    F.countDistinct("date").alias("dias_activo"),
    F.sum(F.when((F.col("date") >= 0)  & (F.col("date") <= 6),  F.col("sum_click"))).alias("clics_semana_1"),
    F.sum(F.when((F.col("date") >= 7)  & (F.col("date") <= 13), F.col("sum_click"))).alias("clics_semana_2"),
    F.sum(F.when((F.col("date") >= 14) & (F.col("date") <= 20), F.col("sum_click"))).alias("clics_semana_3"),
    F.sum(F.when((F.col("date") >= 21) & (F.col("date") <= 27), F.col("sum_click"))).alias("clics_semana_4"),
)

# 4. Clics pre-curso
precurso_agg = vle_precurso.groupBy("id_student", "code_module", "code_presentation").agg(
    F.sum("sum_click").alias("clics_precurso")
)

# 5. Regularidad y tipos_actividad (requiere join con vle para activity_type)
tipos_agg = vle_ventana.join(
    vle.select("id_site", "activity_type"),
    on="id_site", how="left"
).groupBy("id_student", "code_module", "code_presentation").agg(
    F.countDistinct("activity_type").alias("tipos_actividad")
)

# 6. Unir todo con df_activos (left join para conservar estudiantes sin actividad -> quedan como null -> luego a 0)
df_vle = df_activos.join(vle_agg,    on=["id_student", "code_module", "code_presentation"], how="left") \
                   .join(precurso_agg, on=["id_student", "code_module", "code_presentation"], how="left") \
                   .join(tipos_agg,    on=["id_student", "code_module", "code_presentation"], how="left")

# 7. Rellenar nulls con 0 (estudiantes sin ninguna actividad en ventana)
cols_vle = ["total_clics", "dias_activo", "clics_semana_1", "clics_semana_2",
            "clics_semana_3", "clics_semana_4", "clics_precurso", "tipos_actividad"]
df_vle = df_vle.fillna(0, subset=cols_vle)

# 8. Añadir regularidad
df_vle = df_vle.withColumn("regularidad", F.col("dias_activo") / 28.0)

# Verificación final
print(f"df_vle: {df_vle.count():,} filas (debe ser 27.553)")
stats = df_vle.select(cols_vle).describe().toPandas()
stats.iloc[:, 1:] = stats.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").round(3)
display(stats)

Tras dropDuplicates: 9,868,110 filas
df_vle: 27,553 filas (debe ser 27.553)


summary,total_clics,dias_activo,clics_semana_1,clics_semana_2,clics_semana_3,clics_semana_4,clics_precurso,tipos_actividad
count,27553.0,27553.0,27553.0,27553.0,27553.0,27553.0,27553.0,27553.0
mean,267.045,10.653,68.281,61.38,79.445,57.939,72.783,6.483
stddev,322.631,7.193,108.008,92.127,105.204,92.278,133.321,2.575
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
max,6008.0,28.0,4383.0,1550.0,1624.0,2320.0,3400.0,14.0


**Comentarios:**

Tras el proceso de deduplicación exacta quedan 9.868.110 filas en student_vle (-787.170). El join con df_activos devuelve las 27.553 matrículas esperadas y los nulls se rellenan con 0 para conservar en los registros a los alumnos sin actividad en la ventana.

Revisando las estadísticas vemos que la media de total_clics en este caso nos devuelve 267, cifra ligeramente menor que en el EDA (279); la explicación es que se dirigen a poblaciones distintas: en el EDA el cálculo se realiza sobre los 26.304 estudiantes que registraron al menos un clic (los inactivos quedan como NaN y son excluidos). Aquí, el fillna(0) recupera a esos 1.249 alumnos con cero clics, lo que produce una bajada de la media. Ambos valores son correctos, solo que el de el presente notebook refleja el dataset completo que recibe el modelo.

## 3. Features de evaluación

De nuevo aplicamos un join para cruzar en esta ocasión student_assessment con assessments y obtener el módulo y el semestre correspondiente a cada entrega. Filtramos las realizadas dentro de la ventana de predicción con la exclusión de las importadas de otros semestres al no representar actividad real de los alumnos en ese periodo.

Ajustamos nota_media_ventana y nota_min_ventana para que se queden como null cuando el alumno no tiene entregas, ya que contamos con que Catboost gestiona de manera nativa los nulos en variables numéricas.

In [0]:
# 1. Join student_assess + assessments para obtener code_module y code_presentation
assess_join = student_assess.join(
    assessments.select("id_assessment", "code_module", "code_presentation"),
    on="id_assessment",
    how="left"
)

# 2. Filtrar: solo entregas dentro de ventana, no importadas
assess_ventana = assess_join.filter(
    (F.col("date_submitted") <= 27) & (F.col("is_banked") == 0)
)

# 3. Agregar por estudiante-módulo-presentación
assess_agg = assess_ventana.groupBy("id_student", "code_module", "code_presentation").agg(
    F.count("id_assessment").alias("n_entregas_ventana"),
    F.avg("score").alias("nota_media_ventana"),
    F.min("score").alias("nota_min_ventana")
)

# 4. Left join sobre df_vle
df_features = df_vle.join(
    assess_agg,
    on=["id_student", "code_module", "code_presentation"],
    how="left"
)

# 5. entrego_algo + rellenar nulls de conteo (nota se deja null para CatBoost)
df_features = df_features.withColumn(
    "entrego_algo",
    F.when(F.col("n_entregas_ventana") >= 1, 1).otherwise(0)
).fillna(0, subset=["n_entregas_ventana", "entrego_algo"])

# Verificación
print(f"df_features: {df_features.count():,} filas (debe ser 27.553)")
stats = df_features.select("n_entregas_ventana", "entrego_algo", "nota_media_ventana", "nota_min_ventana") \
                   .describe().toPandas()
stats.iloc[:, 1:] = stats.iloc[:, 1:].apply(pd.to_numeric, errors="coerce").round(3)
display(stats)

print("\nEntregas por módulo:")
modulos = df_features.groupBy("code_module").agg(
    F.avg("n_entregas_ventana").alias("media_entregas"),
    F.count(F.when(F.col("nota_media_ventana").isNull(), 1)).alias("nulls_nota")
).orderBy("code_module").toPandas()
modulos["media_entregas"] = modulos["media_entregas"].round(3)
display(modulos)

df_features: 27,553 filas (debe ser 27.553)


summary,n_entregas_ventana,entrego_algo,nota_media_ventana,nota_min_ventana
count,27553.0,27553.0,19782.0,19782.0
mean,0.835,0.718,73.05,72.227
stddev,0.639,0.45,21.737,21.782
min,0.0,0.0,0.0,0.0
max,4.0,1.0,100.0,100.0



Entregas por módulo:


code_module,media_entregas,nulls_nota
AAA,0.903,70
BBB,0.888,742
CCC,0.933,442
DDD,1.003,871
EEE,0.067,2381
FFF,1.192,849
GGG,0.009,2416


**Comentarios:**

Dentro de la ventana de 0 a 27 días un 71,8% de los estudiantes entregaron al menos una tarea. En la tabla de detalle de los nulos y su nota media se aprecia la distribución exacta de los 7.771, que tienen orígenes distintos: por un lado se concentran los de módulos sin evaluación (EEE = 2.381 y GGG = 2.416), algo que es estructural y no depende del estudiante. Por otro lado, en el resto de módulos los null indican que el alumno no entregó nada nuevo dentro de la ventana (ni antes del día 27 ni fuera del filtro is_banked). La decisión orientada a CatBoost es mantener los null, ya que este modelo los trata como categoría propia y la ausencia de nota es en sí señal predictiva.

## 4. Ensamblado final y exportación

Para cerrar el proceso se seleccionan las 25 columnas en el mismo orden que en el df_modelo.csv para garantizar la compatibilidad directa con el pipeline de modelado.

Añadimos también una verificación expresa de ausencia de leakage dejando fuera del output final_result, date_unregistration y date_registration.

In [0]:
# Columnas finales — exactamente las mismas que df_modelo.csv (25 cols)
columnas_modelo = [
    # Identificadores / target
    "id_student", "code_module", "code_presentation", "riesgo",
    # Demográficas
    "gender", "region", "highest_education", "imd_band",
    "age_band", "num_of_prev_attempts", "studied_credits", "disability",
    # VLE
    "total_clics", "dias_activo", "clics_semana_1", "clics_semana_2",
    "clics_semana_3", "clics_semana_4", "clics_precurso",
    "regularidad", "tipos_actividad",
    # Assessment
    "n_entregas_ventana", "entrego_algo", "nota_media_ventana", "nota_min_ventana"
]

df_modelo_spark = df_features.select(columnas_modelo)

# Verificación anti-leakage: estas columnas NO deben estar
cols_prohibidas = ["final_result", "date_unregistration", "date_registration"]
cols_presentes = [c for c in cols_prohibidas if c in df_modelo_spark.columns]
if cols_presentes:
    print(f"⚠️  LEAKAGE DETECTADO: {cols_presentes}")
else:
    print("✅ Sin leakage — columnas prohibidas ausentes")

# Shape final
print(f"Shape: {df_modelo_spark.count():,} filas × {len(df_modelo_spark.columns)} columnas")

# Exportar a CSV en el Workspace
OUTPUT = "/Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/df_modelo_spark.csv"
df_modelo_spark.coalesce(1).write.csv(OUTPUT, header=True, mode="overwrite")
print(f"✅ Exportado en {OUTPUT}")

✅ Sin leakage — columnas prohibidas ausentes
Shape: 27,553 filas × 25 columnas
✅ Exportado en /Workspace/Users/bsolanahurtado@gmail.com/TFM-OULAD/df_modelo_spark.csv


**Comentarios:**

El archivo resultante de 27.553 filas por 25 columnas es idéntico en estructura al df_modelo.csv.

La verificación de leakage nos confirma que ninguna variable que codifique el resultado final ha llegado al dataset de features.

Por último, cabe señalar que la exportación se realiza con coalesce(1) para obtener un csv único en lugar de las particiones por defecto de Spark, lo que simplifica su descarga y uso posterior.